# 01 — Annual Distribution Analysis

Step-by-step analysis of annual generation and revenue distributions for a single site/kind/market.
Reads from the pre-aggregated `generation.duckdb` and `revenue.duckdb` files produced by `forecast-aggregation`.

**Steps:**
1. Config
2. Data loading (GCS or local)
3. Summary stats table
4. Generation histogram + Scott KDE
5. Revenue histogram + KDE + empirical CDF + KS-test
6. Gen-weighted price distribution
7. Historical years overlaid on simulated KDE
8. Percentile comparison table: KDE vs empirical CDF

In [3]:
# ═══════════════════════════════════════════════════════════════════
# Cell 1: Configuration — edit these parameters
# ═══════════════════════════════════════════════════════════════════

# Site / market to analyse
SITE   = "ash_creek_solar"
KIND   = "hub"   # 'hub' or 'node'
MARKET = "da"    # 'da' or 'rt'

# Data source
SOURCE = "gcs"   # 'gcs' or 'local'

# KDE bandwidth method (passed to scipy.stats.gaussian_kde)
KDE_BW_METHOD = "scott"  # 'scott', 'silverman', or a float

# Paths
import os
from pathlib import Path
REPO_ROOT       = Path(os.environ.get("MODEL_GPR_ROOT",
                        "/Users/divy/code/work/infrasure_git_codes/model-gpr")).resolve()
LOCAL_AGG_DIR   = REPO_ROOT / "local_data" / "aggregated_data"
GCS_BUCKET      = "infrasure-model-gpr-data"
GCS_PROJECT     = "infrasure-model-gpr"
GCS_AGG_PREFIX  = "aggregated_data"

print(f"Site: {SITE}  |  Kind: {KIND}  |  Market: {MARKET}  |  Source: {SOURCE}")

Site: ash_creek_solar  |  Kind: hub  |  Market: da  |  Source: gcs


In [4]:
# ═══════════════════════════════════════════════════════════════════
# Cell 2: Data loading
# ═══════════════════════════════════════════════════════════════════

import warnings, tempfile, shutil
import numpy as np
import pandas as pd
import duckdb

warnings.filterwarnings("ignore", message=".*end user credentials.*")
warnings.filterwarnings("ignore", message=".*quota project.*")

def _get_duckdb_as_temp(db_name: str) -> str:
    """Download or copy a DuckDB file to a temp path and return it."""
    tmp = tempfile.mktemp(suffix=".duckdb")
    if SOURCE == "local":
        src = LOCAL_AGG_DIR / SITE / db_name
        if not src.exists():
            raise FileNotFoundError(f"Not found locally: {src}")
        shutil.copy2(str(src), tmp)
    else:
        from google.cloud import storage
        client  = storage.Client(project=GCS_PROJECT)
        bucket  = client.bucket(GCS_BUCKET)
        blob_path = f"{GCS_AGG_PREFIX}/{SITE}/{db_name}"
        blob = bucket.blob(blob_path)
        if not blob.exists():
            raise FileNotFoundError(
                f"Not found in GCS: gs://{GCS_BUCKET}/{blob_path}\n"
                "Run forecast-aggregation first."
            )
        blob.download_to_filename(tmp)
    return tmp

_rev_tmp = _get_duckdb_as_temp("revenue.duckdb")
_gen_tmp = _get_duckdb_as_temp("generation.duckdb")

rc = duckdb.connect(_rev_tmp, read_only=True)
gc = duckdb.connect(_gen_tmp, read_only=True)

# --- Revenue annual (eligible paths only) ---
rev = rc.execute("""
    SELECT path_id, segment, source_year,
           annual_revenue_usd,
           price_per_mwh_gen_weighted,
           price_per_mwh_simple_mean,
           revenue_coverage_pct,
           eligible_for_rev_dist
    FROM annual
    WHERE kind=? AND market=?
    ORDER BY path_id
""", [KIND, MARKET]).df()

# --- Generation annual (eligible paths only) ---
gen = gc.execute("""
    SELECT path_id, segment, source_year,
           annual_generation_mwh,
           gen_coverage_pct,
           eligible_for_gen_dist
    FROM annual
    WHERE kind=? AND market=?
    ORDER BY path_id
""", [KIND, MARKET]).df()

rc.close(); gc.close()
import os as _os
for _f in [_rev_tmp, _gen_tmp]:
    try: _os.unlink(_f)
    except: pass

# Eligible subsets
rev_elig = rev[rev["eligible_for_rev_dist"] == True].copy()
gen_elig = gen[gen["eligible_for_gen_dist"] == True].copy()

print(f"Revenue: {len(rev_elig)} eligible paths  ({len(rev) - len(rev_elig)} excluded)")
print(f"Generation: {len(gen_elig)} eligible paths  ({len(gen) - len(gen_elig)} excluded)")
print()
print("Eligible revenue paths by segment:")
print(rev_elig["segment"].value_counts().to_string())

Revenue: 114 eligible paths  (33 excluded)
Generation: 146 eligible paths  (1 excluded)

Eligible revenue paths by segment:
segment
simulated     100
historical     14


In [5]:
# ═══════════════════════════════════════════════════════════════════
# Cell 3: Summary statistics table
# ═══════════════════════════════════════════════════════════════════

from IPython.display import display, HTML

def _pct_stats(series: pd.Series, scale=1.0, unit="") -> dict:
    s = series.dropna() * scale
    return {
        "N": len(s),
        "Min": f"{s.min():.2f} {unit}",
        "P10": f"{s.quantile(0.10):.2f} {unit}",
        "P25": f"{s.quantile(0.25):.2f} {unit}",
        "Median": f"{s.quantile(0.50):.2f} {unit}",
        "Mean": f"{s.mean():.2f} {unit}",
        "P75": f"{s.quantile(0.75):.2f} {unit}",
        "P90": f"{s.quantile(0.90):.2f} {unit}",
        "Max": f"{s.max():.2f} {unit}",
    }

stats_rows = {
    "Generation (GWh)": _pct_stats(gen_elig["annual_generation_mwh"], scale=1/1e3, unit="GWh"),
    "Revenue (M USD)": _pct_stats(rev_elig["annual_revenue_usd"], scale=1/1e6, unit="M$"),
    "Price gen-wtd ($/MWh)": _pct_stats(rev_elig["price_per_mwh_gen_weighted"], unit="$/MWh"),
}

stats_df = pd.DataFrame(stats_rows).T
print(f"\n{'═'*70}")
print(f"  {SITE}  |  {KIND}/{MARKET}  |  source: {SOURCE}")
print(f"{'═'*70}")
display(stats_df)

# Excluded paths breakdown
excl_rev = rev[rev["eligible_for_rev_dist"] == False]
no_price = excl_rev[excl_rev["revenue_coverage_pct"] < 5].shape[0]
partial  = excl_rev[excl_rev["revenue_coverage_pct"] >= 5].shape[0]
print(f"\nExcluded from revenue distribution: {len(excl_rev)} paths")
print(f"  - No price data (rev_cov < 5%):   {no_price}")
print(f"  - Partial year  (5% ≤ rev_cov < 95%): {partial}")


══════════════════════════════════════════════════════════════════════
  ash_creek_solar  |  hub/da  |  source: gcs
══════════════════════════════════════════════════════════════════════


,N,Min,P10,P25,Median,Mean,P75,P90,Max
Generation (GWh),146,698.88 GWh,724.85 GWh,737.40 GWh,753.52 GWh,752.27 GWh,763.39 GWh,779.75 GWh,819.75 GWh
Revenue (M USD),114,19.65 M$,23.74 M$,25.22 M$,27.57 M$,28.15 M$,30.38 M$,33.93 M$,43.24 M$
Price gen-wtd ($/MWh),114,24.92 $/MWh,31.71 $/MWh,33.62 $/MWh,36.74 $/MWh,37.26 $/MWh,40.18 $/MWh,44.54 $/MWh,54.50 $/MWh



Excluded from revenue distribution: 33 paths
  - No price data (rev_cov < 5%):   31
  - Partial year  (5% ≤ rev_cov < 95%): 2


In [6]:
# ═══════════════════════════════════════════════════════════════════
# Cell 4: Generation histogram + Scott KDE
# ═══════════════════════════════════════════════════════════════════

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde

gen_sim  = gen_elig[gen_elig["segment"] == "simulated"]["annual_generation_mwh"].values / 1e3
gen_hist = gen_elig[gen_elig["segment"] == "historical"]["annual_generation_mwh"].values / 1e3
gen_all  = gen_elig["annual_generation_mwh"].values / 1e3

# Scott KDE over full data
kde_gen = gaussian_kde(gen_all, bw_method=KDE_BW_METHOD)
x_gen   = np.linspace(gen_all.min() * 0.97, gen_all.max() * 1.03, 300)
y_gen   = kde_gen(x_gen)

# Percentile markers
p10_g, p50_g, p90_g = np.percentile(gen_all, [10, 50, 90])

fig_gen = go.Figure()

# Simulated histogram
if len(gen_sim) > 0:
    fig_gen.add_trace(go.Histogram(
        x=gen_sim, name="Simulated",
        histnorm="probability density",
        marker_color="rgba(99,110,250,0.55)",
        nbinsx=20, showlegend=True
    ))
# Historical histogram
if len(gen_hist) > 0:
    fig_gen.add_trace(go.Histogram(
        x=gen_hist, name="Historical",
        histnorm="probability density",
        marker_color="rgba(0,204,150,0.65)",
        nbinsx=15, showlegend=True
    ))
# KDE line
fig_gen.add_trace(go.Scatter(
    x=x_gen, y=y_gen,
    name=f"KDE (Scott bw={kde_gen.factor:.3f})",
    mode="lines", line=dict(color="#ab63fa", width=2.5)
))
# Percentile lines
for pval, plabel, pcolor in [(p10_g, "P10", "#636efa"), (p50_g, "P50", "#ef553b"), (p90_g, "P90", "#636efa")]:
    fig_gen.add_vline(x=pval, line_dash="dash", line_color=pcolor, line_width=1.5,
                      annotation_text=f"{plabel}: {pval:.1f}", annotation_position="top")

fig_gen.update_layout(
    title=f"Annual Generation Distribution — {SITE}  {KIND}/{MARKET}",
    xaxis_title="Annual Generation (GWh)",
    yaxis_title="Probability Density",
    barmode="overlay", template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
    width=800, height=480,
)
fig_gen.show()
print(f"\nScott bandwidth factor: {kde_gen.factor:.4f}")
print(f"P10: {p10_g:.1f} GWh  |  P50: {p50_g:.1f} GWh  |  P90: {p90_g:.1f} GWh")


Scott bandwidth factor: 0.3691
P10: 724.8 GWh  |  P50: 753.5 GWh  |  P90: 779.7 GWh


In [7]:
# ═══════════════════════════════════════════════════════════════════
# Cell 5: Revenue histogram + KDE + empirical CDF + KS-test
# ═══════════════════════════════════════════════════════════════════

from scipy.stats import gaussian_kde, ks_2samp

rev_sim  = rev_elig[rev_elig["segment"] == "simulated"]["annual_revenue_usd"].values / 1e6
rev_hist = rev_elig[rev_elig["segment"] == "historical"]["annual_revenue_usd"].values / 1e6
rev_all  = rev_elig["annual_revenue_usd"].values / 1e6

# Scott KDE
kde_rev  = gaussian_kde(rev_all, bw_method=KDE_BW_METHOD)
x_rev    = np.linspace(rev_all.min() * 0.94, rev_all.max() * 1.06, 400)
y_rev    = kde_rev(x_rev)

# CDF: empirical (numpy) vs KDE integrated
rev_sorted = np.sort(rev_all)
empirical_cdf = np.arange(1, len(rev_sorted) + 1) / len(rev_sorted)

# KDE CDF: cumulative trapezoid integration
from scipy.integrate import cumulative_trapezoid
y_cdf_kde = cumulative_trapezoid(y_rev, x_rev, initial=0)
y_cdf_kde = y_cdf_kde / y_cdf_kde[-1]  # normalise to [0, 1]

# KS test: draw samples from KDE and compare against observed data
np.random.seed(42)
kde_samples = kde_rev.resample(2000).flatten()
ks_stat, ks_pval = ks_2samp(rev_all, kde_samples)

# Figure: side-by-side PDF | CDF
fig_rev = make_subplots(
    rows=1, cols=2,
    subplot_titles=["PDF — Revenue", "CDF — KDE vs Empirical"],
    horizontal_spacing=0.12
)

# — PDF panel —
if len(rev_sim) > 0:
    fig_rev.add_trace(go.Histogram(
        x=rev_sim, name="Simulated",
        histnorm="probability density",
        marker_color="rgba(99,110,250,0.5)", nbinsx=20
    ), row=1, col=1)
if len(rev_hist) > 0:
    fig_rev.add_trace(go.Histogram(
        x=rev_hist, name="Historical",
        histnorm="probability density",
        marker_color="rgba(0,204,150,0.65)", nbinsx=12
    ), row=1, col=1)
fig_rev.add_trace(go.Scatter(
    x=x_rev, y=y_rev,
    name=f"Scott KDE (bw={kde_rev.factor:.3f})",
    mode="lines", line=dict(color="#ab63fa", width=2.5)
), row=1, col=1)

# P10/P50/P90 on PDF
for pval, plabel in zip(np.percentile(rev_all, [10, 50, 90]), ["P10", "P50", "P90"]):
    fig_rev.add_vline(x=pval, line_dash="dot", line_color="#636efa", line_width=1.2,
                      annotation_text=plabel, row=1, col=1)

# — CDF panel —
fig_rev.add_trace(go.Scatter(
    x=rev_sorted, y=empirical_cdf,
    name="Empirical CDF (numpy)",
    mode="lines", line=dict(color="#00cc96", width=2, dash="dash")
), row=1, col=2)
fig_rev.add_trace(go.Scatter(
    x=x_rev, y=y_cdf_kde,
    name="KDE CDF (Scott)",
    mode="lines", line=dict(color="#ab63fa", width=2.5)
), row=1, col=2)

fig_rev.update_layout(
    title=f"Annual Revenue — {SITE}  {KIND}/{MARKET}  ({len(rev_all)} eligible paths)",
    xaxis_title="Revenue (M USD)", yaxis_title="Density",
    xaxis2_title="Revenue (M USD)", yaxis2_title="Cumulative Probability",
    barmode="overlay", template="plotly_white",
    legend=dict(orientation="h", y=-0.18), width=1050, height=480,
)
fig_rev.show()

# KS-test summary
print(f"\n{'─'*55}")
print(f"  KS-test: KDE samples (n=2000) vs observed data (n={len(rev_all)})")
print(f"  D-statistic : {ks_stat:.4f}")
print(f"  p-value     : {ks_pval:.4f}")
if ks_pval > 0.05:
    print(f"  Interpretation: KDE is a good fit (fail to reject H0 at 5%)")
else:
    print(f"  Interpretation: KDE differs from empirical at 5% significance")
print(f"{'─'*55}")


───────────────────────────────────────────────────────
  KS-test: KDE samples (n=2000) vs observed data (n=114)
  D-statistic : 0.0620
  p-value     : 0.7764
  Interpretation: KDE is a good fit (fail to reject H0 at 5%)
───────────────────────────────────────────────────────


In [8]:
# ═══════════════════════════════════════════════════════════════════
# Cell 6: Gen-weighted price distribution
# ═══════════════════════════════════════════════════════════════════

price_all  = rev_elig["price_per_mwh_gen_weighted"].dropna().values
price_sim  = rev_elig[rev_elig["segment"] == "simulated"]["price_per_mwh_gen_weighted"].dropna().values
price_hist = rev_elig[rev_elig["segment"] == "historical"]["price_per_mwh_gen_weighted"].dropna().values

kde_price = gaussian_kde(price_all, bw_method=KDE_BW_METHOD)
x_price   = np.linspace(price_all.min() * 0.94, price_all.max() * 1.06, 300)
y_price   = kde_price(x_price)

price_sorted     = np.sort(price_all)
price_ecdf       = np.arange(1, len(price_sorted) + 1) / len(price_sorted)
price_kde_cdf    = cumulative_trapezoid(y_price, x_price, initial=0)
price_kde_cdf   /= price_kde_cdf[-1]

kde_price_samples = kde_price.resample(2000).flatten()
ks_p_stat, ks_p_pval = ks_2samp(price_all, kde_price_samples)

fig_price = make_subplots(rows=1, cols=2,
    subplot_titles=["PDF — Gen-weighted Price", "CDF — KDE vs Empirical"],
    horizontal_spacing=0.12)

if len(price_sim) > 0:
    fig_price.add_trace(go.Histogram(
        x=price_sim, name="Simulated",
        histnorm="probability density",
        marker_color="rgba(99,110,250,0.5)", nbinsx=20
    ), row=1, col=1)
if len(price_hist) > 0:
    fig_price.add_trace(go.Histogram(
        x=price_hist, name="Historical",
        histnorm="probability density",
        marker_color="rgba(0,204,150,0.65)", nbinsx=12
    ), row=1, col=1)
fig_price.add_trace(go.Scatter(
    x=x_price, y=y_price,
    name=f"Scott KDE (bw={kde_price.factor:.3f})",
    mode="lines", line=dict(color="#ab63fa", width=2.5)
), row=1, col=1)
for pval, plabel in zip(np.percentile(price_all, [10, 50, 90]), ["P10", "P50", "P90"]):
    fig_price.add_vline(x=pval, line_dash="dot", line_color="#636efa", line_width=1.2,
                        annotation_text=plabel, row=1, col=1)

fig_price.add_trace(go.Scatter(
    x=price_sorted, y=price_ecdf,
    name="Empirical CDF (numpy)",
    mode="lines", line=dict(color="#00cc96", width=2, dash="dash")
), row=1, col=2)
fig_price.add_trace(go.Scatter(
    x=x_price, y=price_kde_cdf,
    name="KDE CDF (Scott)",
    mode="lines", line=dict(color="#ab63fa", width=2.5)
), row=1, col=2)

fig_price.update_layout(
    title=f"Annual Gen-weighted Price — {SITE}  {KIND}/{MARKET}",
    xaxis_title="$/MWh", yaxis_title="Density",
    xaxis2_title="$/MWh", yaxis2_title="Cumulative Probability",
    barmode="overlay", template="plotly_white",
    legend=dict(orientation="h", y=-0.18), width=1050, height=480,
)
fig_price.show()

print(f"\n{'─'*55}")
print(f"  Price KS-test: D={ks_p_stat:.4f}  p={ks_p_pval:.4f}")
print(f"  P50 gen-wtd price: ${np.median(price_all):.2f}/MWh")
print(f"  Price spread P10–P90: ${np.percentile(price_all,10):.2f} – ${np.percentile(price_all,90):.2f}/MWh")
print(f"{'─'*55}")


───────────────────────────────────────────────────────
  Price KS-test: D=0.0743  p=0.5635
  P50 gen-wtd price: $36.74/MWh
  Price spread P10–P90: $31.71 – $44.54/MWh
───────────────────────────────────────────────────────


In [9]:
# ═══════════════════════════════════════════════════════════════════
# Cell 7: Historical years overlaid on simulated KDE
#         + Recent years highlighted + mean comparison
# ═══════════════════════════════════════════════════════════════════
N_RECENT = 3  # last N historical years highlighted with distinct color

rev_median = np.median(rev_all)

# IQR outlier bounds from simulated only
if len(rev_sim) >= 4:
    q1_sim, q3_sim = np.percentile(rev_sim, [25, 75])
    iqr_sim   = q3_sim - q1_sim
    lo_bound  = q1_sim - 1.5 * iqr_sim
    hi_bound  = q3_sim + 1.5 * iqr_sim
else:
    lo_bound, hi_bound = -np.inf, np.inf

fig_overlay = go.Figure()

# Simulated KDE shaded area
if len(rev_sim) > 1:
    kde_sim_only = gaussian_kde(rev_sim, bw_method=KDE_BW_METHOD)
    x_sim_kde    = np.linspace(rev_sim.min() * 0.93, rev_sim.max() * 1.07, 350)
    y_sim_kde    = kde_sim_only(x_sim_kde)
    fig_overlay.add_trace(go.Scatter(
        x=np.concatenate([x_sim_kde, x_sim_kde[::-1]]),
        y=np.concatenate([y_sim_kde, np.zeros(len(y_sim_kde))]),
        fill="toself",
        fillcolor="rgba(99,110,250,0.15)",
        line=dict(color="rgba(99,110,250,0.0)"),
        name="Simulated KDE area", showlegend=True
    ))
    fig_overlay.add_trace(go.Scatter(
        x=x_sim_kde, y=y_sim_kde,
        name=f"Simulated KDE",
        mode="lines", line=dict(color="#636efa", width=2)
    ))

# P50 line (all eligible)
fig_overlay.add_vline(x=rev_median, line_dash="dash", line_color="#ef553b",
                      line_width=1.5, annotation_text=f"P50 all: {rev_median:.1f}",
                      annotation_position="top right")

# Historical years as scatter points
hist_rev = rev_elig[rev_elig["segment"] == "historical"].copy()
hist_rev["rev_M"] = hist_rev["annual_revenue_usd"] / 1e6

for _, row in hist_rev.iterrows():
    is_outlier = (row["rev_M"] < lo_bound) or (row["rev_M"] > hi_bound)
    above_p50  = row["rev_M"] > rev_median
    color      = "#ef553b" if is_outlier else ("#00cc96" if above_p50 else "#636efa")
    symbol     = "star" if is_outlier else "circle"
    kde_y_val  = float(kde_rev(np.array([row["rev_M"]]))[0]) if len(rev_all) > 1 else 0
    fig_overlay.add_trace(go.Scatter(
        x=[row["rev_M"]], y=[kde_y_val],
        mode="markers+text",
        text=[str(int(row["source_year"]))],
        textposition="top center",
        marker=dict(color=color, size=10 if is_outlier else 8, symbol=symbol),
        name=f"{int(row['source_year'])} {'★ outlier' if is_outlier else ''}",
        showlegend=False
    ))

# IQR bounds
if lo_bound > -np.inf:
    fig_overlay.add_vrect(x0=lo_bound, x1=hi_bound,
        fillcolor="rgba(0,204,150,0.06)", line_width=0,
        annotation_text="IQR×1.5 bounds", annotation_position="top left")

# ── Simulated mean vline ──
sim_mean_rev  = float(rev_elig[rev_elig["segment"] == "simulated"]["annual_revenue_usd"].mean() / 1e6)
hist_mean_rev = float(rev_elig[rev_elig["segment"] == "historical"]["annual_revenue_usd"].mean() / 1e6)
fig_overlay.add_vline(x=sim_mean_rev,  line_dash="solid", line_color="#636efa", line_width=1.5,
                      annotation_text=f"Sim mean: {sim_mean_rev:.1f}", annotation_position="top left")
fig_overlay.add_vline(x=hist_mean_rev, line_dash="solid", line_color="#00cc96", line_width=1.5,
                      annotation_text=f"Hist mean: {hist_mean_rev:.1f}", annotation_position="top right")

# ── Highlight recent N years with distinct star markers ──
recent_hist = hist_rev.sort_values("source_year").tail(N_RECENT)
for i, (_, row) in enumerate(recent_hist.iterrows()):
    kde_y_val = float(kde_rev(np.array([row["rev_M"]]))[0]) if len(rev_all) > 1 else 0
    fig_overlay.add_trace(go.Scatter(
        x=[row["rev_M"]], y=[kde_y_val * 1.18],
        mode="markers+text",
        text=[f"★{int(row['source_year'])}"],
        textposition="top center",
        textfont=dict(color="#d62728", size=11, family="Arial Black"),
        marker=dict(color="#d62728", size=13, symbol="star"),
        name=f"{int(row['source_year'])} (recent)",
        showlegend=True,
    ))

fig_overlay.update_layout(
    title=f"Historical Years on Simulated KDE — {SITE}  {KIND}/{MARKET}<br>"
          f"<sup>Red ★ = recent {N_RECENT} years | Red/star = IQR outlier | Blue/green = below/above P50 | "
          f"Vertical lines = simulated mean (blue) vs historical mean (green)</sup>",
    xaxis_title="Annual Revenue (M USD)",
    yaxis_title="KDE Density",
    template="plotly_white", width=960, height=520,
)
fig_overlay.show()

# Print IQR outliers
outliers = hist_rev[
    (hist_rev["rev_M"] < lo_bound) | (hist_rev["rev_M"] > hi_bound)
][["source_year", "rev_M", "price_per_mwh_gen_weighted"]]
print(f"\nHistorical IQR outliers ({len(outliers)}):")
for _, r in outliers.iterrows():
    print(f"  {int(r['source_year'])}: ${r['rev_M']:.2f}M  "
          f"(gen-wtd price ${r['price_per_mwh_gen_weighted']:.1f}/MWh)")

# Print mean comparison
print(f"\nMean annual revenue:")
print(f"  Simulated mean  : ${sim_mean_rev:.2f}M")
print(f"  Historical mean : ${hist_mean_rev:.2f}M")
print(f"  Difference      : ${sim_mean_rev - hist_mean_rev:+.2f}M  ({(sim_mean_rev/hist_mean_rev-1)*100:+.1f}%)")

# Print recent years trend vs P50
print(f"\nLast {N_RECENT} historical years vs simulated P50 (${rev_median:.2f}M):")
for _, r in recent_hist.iterrows():
    diff = r["rev_M"] - rev_median
    arrow = "↑" if diff > 0 else "↓"
    print(f"  {int(r['source_year'])}: ${r['rev_M']:.2f}M  {arrow} ${abs(diff):.2f}M vs P50"
          f"  (gen-wtd ${r['price_per_mwh_gen_weighted']:.1f}/MWh)")


Historical IQR outliers (2):
  2011: $40.93M  (gen-wtd price $50.6/MWh)
  2022: $43.24M  (gen-wtd price $54.5/MWh)

Mean annual revenue:
  Simulated mean  : $28.16M
  Historical mean : $28.08M
  Difference      : $+0.08M  (+0.3%)

Last 3 historical years vs simulated P50 ($27.57M):
  2022: $43.24M  ↑ $15.67M vs P50  (gen-wtd $54.5/MWh)
  2023: $32.85M  ↑ $5.28M vs P50  (gen-wtd $43.3/MWh)
  2024: $19.65M  ↓ $7.91M vs P50  (gen-wtd $24.9/MWh)


In [10]:
# ═══════════════════════════════════════════════════════════════════
# Cell 8: Percentile comparison table — KDE vs empirical CDF
# ═══════════════════════════════════════════════════════════════════

# Percentiles to compare
percentiles = [5, 10, 25, 50, 75, 90, 95]

# Empirical: direct numpy percentile on observed data
empirical_vals = {p: np.percentile(rev_all, p) for p in percentiles}

# KDE: invert KDE CDF (find x where CDF = p/100)
y_cdf_full = cumulative_trapezoid(y_rev, x_rev, initial=0)
y_cdf_full = y_cdf_full / y_cdf_full[-1]
kde_vals   = {p: float(np.interp(p / 100.0, y_cdf_full, x_rev)) for p in percentiles}

comparison_rows = []
for p in percentiles:
    emp_v = empirical_vals[p]
    kde_v = kde_vals[p]
    diff  = kde_v - emp_v
    comparison_rows.append({
        "Percentile": f"P{p}",
        "Empirical (M$)": f"{emp_v:.3f}",
        "KDE / Scott (M$)": f"{kde_v:.3f}",
        "Diff (M$)": f"{diff:+.3f}",
        "Diff (%)": f"{100*diff/emp_v:+.2f}%" if emp_v != 0 else "—",
    })

cmp_df = pd.DataFrame(comparison_rows).set_index("Percentile")

print(f"\n{'═'*65}")
print(f"  Percentile comparison: KDE (Scott) vs Empirical CDF")
print(f"  Site: {SITE}  |  {KIND}/{MARKET}  |  N={len(rev_all)} eligible paths")
print(f"{'═'*65}")
display(cmp_df)

print(f"\nKS test (KDE samples vs data):  D={ks_stat:.4f}  p={ks_pval:.4f}")
print("Note: KDE smooths the tails slightly — P5 and P95 may differ from empirical.")
print("For risk/financial use, empirical quantiles are the most conservative choice.")

# Also plot CDF comparison with percentile markers
fig_cmp = go.Figure()
fig_cmp.add_trace(go.Scatter(
    x=rev_sorted, y=empirical_cdf, name="Empirical CDF",
    mode="lines", line=dict(color="#00cc96", width=2, dash="dash")
))
fig_cmp.add_trace(go.Scatter(
    x=x_rev, y=y_cdf_kde, name="KDE CDF (Scott)",
    mode="lines", line=dict(color="#ab63fa", width=2.5)
))
for p in [10, 50, 90]:
    fig_cmp.add_hline(y=p/100, line_dash="dot", line_color="#aaa", line_width=1,
                      annotation_text=f"P{p}", annotation_position="left")
fig_cmp.update_layout(
    title=f"CDF Comparison — Revenue  |  {SITE}  {KIND}/{MARKET}",
    xaxis_title="Annual Revenue (M USD)", yaxis_title="Cumulative Probability",
    template="plotly_white", width=750, height=430,
    legend=dict(orientation="h", y=-0.18),
)
fig_cmp.show()


═════════════════════════════════════════════════════════════════
  Percentile comparison: KDE (Scott) vs Empirical CDF
  Site: ash_creek_solar  |  hub/da  |  N=114 eligible paths
═════════════════════════════════════════════════════════════════


,Empirical (M$),KDE / Scott (M$),Diff (M$),Diff (%)
Percentile,,,,
P5,21.843,21.577,-0.267,-1.22%
P10,23.740,22.851,-0.888,-3.74%
P25,25.222,25.013,-0.209,-0.83%
P50,27.566,27.697,+0.131,+0.48%
P75,30.383,30.822,+0.439,+1.45%
P90,33.929,34.083,+0.155,+0.46%
P95,36.052,36.622,+0.569,+1.58%



KS test (KDE samples vs data):  D=0.0620  p=0.7764
Note: KDE smooths the tails slightly — P5 and P95 may differ from empirical.
For risk/financial use, empirical quantiles are the most conservative choice.
